# Leslie Matrices — Part 1 Companion Notebook



This notebook accompanies the blog post **Of Life, Death and Linear Algebra (Part 1)**.



## Learning goals

- Build a Leslie matrix from fertility and survival assumptions

- Project an age-structured population forward in time

- Connect simulations to eigenvalues and stable age distributions



## How to use this notebook

Run the cells in order. After each section, pause and compare the numerical output with the biological interpretation: who is being born, who is aging, and what pattern is emerging over time?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# 3-class Leslie matrix
L = np.array([
    [0.0, 1.2, 0.0],
    [0.8, 0.0, 0.0],
    [0.0, 0.7, 0.0]
])

x0 = np.array([1.0, 1.0, 1.0])
L, x0

## Step 1 — Encode the demographic assumptions



The matrix below represents a three-class population.



- The first row contains fertility rates.

- The subdiagonal contains survival transitions into the next class.

- All other entries are zero because individuals only move forward one class at a time.



Before running the next cell, ask: which class is producing births, and which classes are only aging?

In [ ]:
def project_population(L, x0, T=30):
    X = np.zeros((T + 1, len(x0)))
    X[0] = x0
    for t in range(T):
        X[t + 1] = L @ X[t]
    return X

X = project_population(L, x0, T=40)
X[:5]

## Step 2 — Project the population forward



The helper function applies the same Leslie matrix repeatedly. This is the computational version of the recurrence



$$

\mathbf{x}(t+1)=L\mathbf{x}(t).

$$



Look at the first few rows of the output and identify when the influence of the initial population vector starts to fade.

In [ ]:
# Plot age classes over time
plt.figure(figsize=(8, 4))
for i in range(X.shape[1]):
    plt.plot(X[:, i], label=f'Class {i+1}')
plt.title('Population by Age Class')
plt.xlabel('Time step')
plt.ylabel('Population')
plt.legend()
plt.tight_layout()
plt.show()

## Step 3 — Connect simulation to theory



The next computation extracts the dominant eigenvalue and its eigenvector.



- The dominant eigenvalue estimates the long-run growth factor per time step.

- The associated eigenvector gives the stable age composition once we normalize it.



This is the key bridge between short-run simulation and long-run demographic structure.

In [ ]:
# Eigen analysis
eigvals, eigvecs = np.linalg.eig(L)
idx = np.argmax(eigvals.real)
lambda1 = eigvals[idx].real
stable = eigvecs[:, idx].real
stable = stable / stable.sum()

print(f'Dominant eigenvalue lambda1 = {lambda1:.4f}')
print('Stable age distribution:', np.round(stable, 4))

In [ ]:
# Convergence of normalized age composition
shares = X / X.sum(axis=1, keepdims=True)

plt.figure(figsize=(8, 4))
for i in range(shares.shape[1]):
    plt.plot(shares[:, i], label=f'Share class {i+1}')
for i, s in enumerate(stable):
    plt.axhline(s, linestyle='--', linewidth=1)
plt.title('Convergence to Stable Age Distribution')
plt.xlabel('Time step')
plt.ylabel('Share')
plt.legend()
plt.tight_layout()
plt.show()

## Exercises and reflections



1. Change one fertility value and predict the effect on the dominant eigenvalue before running the cell again.

2. Start from a new initial vector and compare the short-run path with the original simulation.

3. Increase one survival coefficient and decide whether the transient effect and long-run effect point in the same direction.

4. Explain in words why the normalized population shares converge even when total population size changes.

5. If this model represented a real species, which additional age class would you add first, and why?